# Calculate outcomes and admissions by LSOA


__Results__:

For the base scenario and with adding MT to both Taunton and Worcester, and for including and excluding lossy drip-and-ship transfers, save these files:
+ `results_full_{scenario_name}.csv`
+ `results_stats_{scenario_name}.csv`
+ `results_stats_by_unit_{scenario_name}.csv`

## Code setup

In [1]:
import os
import numpy as np
import pandas as pd

from classes.geography_processing import Geoprocessing
from classes.model import Model
from classes.scenario import Scenario
from classes.add_net_benefit import add_net_benefit, add_lossy_net_benefit

In [2]:
dir_output = './output'

## Load data

Scenario parameters:

In [3]:
p = './output/scenario_parameters.csv'

# Use converters to turn units column from strings to lists.
# Assumes that no postcodes contain spaces!
converters = {
    'new_mt_units': lambda x: (
        x.replace("'", '').replace(',', '')[1:-1].split()),
}
df_scen = pd.read_csv(p, converters=converters)

In [4]:
display(df_scen.head(6))

,scenario_name,scenario_geoprocess_name,new_mt_units,single_unit_changed,prop_lvo_dripship_treated_with_mt,mt_travel_bias_minutes,prop_lvo
0,base,base,[],0,1.0,0.1,0.3
1,base_lossydripshipmt,base,[],0,0.6,0.1,0.3
2,taunworc,taunworc,"[TA15DA, WR51DD]",0,1.0,0.1,0.3
3,taunworc_lossydripshipmt,taunworc,"[TA15DA, WR51DD]",0,0.6,0.1,0.3
4,only_RG15AN,only_RG15AN,[RG15AN],1,1.0,0.1,0.3
5,only_RG15AN_lossydripshipmt,only_RG15AN,[RG15AN],1,0.6,0.1,0.3


Split into scenarios to calculate in full and scenarios to only calculate as summary:

In [5]:
m = df_scen['single_unit_changed'] == 0
df_scen_full = df_scen.loc[m].drop('single_unit_changed', axis='columns')
df_scen_sing = df_scen.loc[~m].drop('single_unit_changed', axis='columns')

In [6]:
display(df_scen_full)

,scenario_name,scenario_geoprocess_name,new_mt_units,prop_lvo_dripship_treated_with_mt,mt_travel_bias_minutes,prop_lvo
0,base,base,[],1.0,0.1,0.3
1,base_lossydripshipmt,base,[],0.6,0.1,0.3
2,taunworc,taunworc,"[TA15DA, WR51DD]",1.0,0.1,0.3
3,taunworc_lossydripshipmt,taunworc,"[TA15DA, WR51DD]",0.6,0.1,0.3


## Main calculations

In [7]:
def calculate_geo_and_outcomes(
        mt_travel_bias,
        new_mt_units,
        options_prop_lvo_dripship_treated_with_mt
    ):
    # Process geographic data (only needed when hospital data changes):
    geo = Geoprocessing(limit_to_england=True,
                        mt_travel_bias=mt_travel_bias,
                        new_mt_centres=new_mt_units)
    geo.run()
    geodata = geo.combined_data.copy().reset_index()
    
    # Set up scenario
    scenario = Scenario({
        'name': 'temporary_name',
        'limit_to_england' : True
    })
    
    # Set up model
    model = Model(
        scenario=scenario,
        geodata=geodata
    )
    
    # Run model
    model.run()
    results = model.full_results

    # Add net benefit column
    results = add_net_benefit(
        results,
        prop_lvo=prop_lvo,
    )
    for p in options_prop_lvo_dripship_treated_with_mt:
        results = add_lossy_net_benefit(
            results, prop_lvo=prop_lvo, prop_lvo_ds_mt=p)

    # Round outcome results:
    cols_mrs = [c for c in results.columns if (
                ('mrs' in c) | ('improvement' in c))]
    cols_util = [c for c in results.columns if ('util' in c)]
    results[cols_mrs] = results[cols_mrs].round(3)
    results[cols_util] = results[cols_util].round(5)
    
    return results

## Run for all scenarios: main changes

Save a copy of the full results to file.

For each set of changed units, calculate the new geography and outcomes data. Then for each set of stroke population parameters, combine the outcomes into the full population.

In [8]:
for scen_geo_name, df in df_scen_full.groupby('scenario_geoprocess_name'):
    # These parameters should be the same for all df in group:
    mt_travel_bias = df.iloc[0]['mt_travel_bias_minutes']
    new_mt_units = df.iloc[0]['new_mt_units']
    prop_lvo = df.iloc[0]['prop_lvo']
    
    options_prop_lvo_dripship_treated_with_mt = (
        df['prop_lvo_dripship_treated_with_mt'].unique())
    if len(df) != len(options_prop_lvo_dripship_treated_with_mt):
        print('Check this!!')
    # Shared geography and outcome results:
    results = calculate_geo_and_outcomes(
        mt_travel_bias,
        new_mt_units,
        options_prop_lvo_dripship_treated_with_mt
    )

    # Get descriptive stats for all numeric columns
    descriptive_stats = results.describe(percentiles=[0.25, 0.5, 0.75]).T
    descriptive_stats = descriptive_stats.round(5)

    # Repeat descriptive stats, grouped by nearest_ivt_unit
    results_stats_by_unit = results.groupby('nearest_ivt_unit').describe().T
    # Unpack index column so two-level index becomes two columns
    results_stats_by_unit = results_stats_by_unit.reset_index()
    rename_dict = {'level_0': 'variable', 'level_1': 'statistic'}
    results_stats_by_unit = results_stats_by_unit.rename(columns=rename_dict)
    results_stats_by_unit = results_stats_by_unit.round(5)

    # Save results:
    f = f'results_full_{scen_geo_name}.csv'
    results.to_csv(os.path.join(dir_output, f))

    f = f'results_stats_{scen_geo_name}.csv'
    descriptive_stats.to_csv(os.path.join(dir_output, f))

    f = f'results_stats_by_unit_{scen_geo_name}.csv'
    results_stats_by_unit.to_csv(os.path.join(dir_output, f))

The input mRS distributions might be improperly labelled.
The input mRS distributions might be improperly labelled.
The input mRS distributions might be improperly labelled.
The input mRS distributions might be improperly labelled.


## Conclusion

Calculated full results, descriptive stats, and stats by unit and saved to file.